In [4]:
# Install libraries
!pip install pandas numpy scikit-learn matplotlib

In [5]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

print("Libraries loaded successfully")

Libraries loaded successfully


In [10]:
import pandas as pd

df = pd.read_csv("student-por.csv", sep=";")

print(df.shape)
df.head()

(649, 33)


,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,4,3,4,1,1,3,4,0,11,11
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,5,3,3,1,1,3,2,9,11,11
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,4,3,2,2,3,3,6,12,13,12
3,GP,F,15,U,GT3,T,4,2,health,services,...,3,2,2,1,1,5,0,14,14,14
4,GP,F,16,U,GT3,T,3,3,other,other,...,4,3,2,1,2,5,0,11,13,13


In [13]:
# Create classification target from final grade G3
def categorize_grade(g):
    if g < 10:
        return 0   # Low
    elif g < 15:
        return 1   # Medium
    else:
        return 2   # High

df["target"] = df["G3"].apply(categorize_grade)

print(df["target"].value_counts())

target
1    418
2    131
0    100
Name: count, dtype: int64


In [19]:
# Prepare features and target (FIXED - no data leakage)

# Remove grade columns (G1, G2, G3) and target
X = df.drop(columns=["G1", "G2", "G3", "target"])
y = df["target"]

# Convert categorical variables into numeric
X = pd.get_dummies(X)

# Scale features
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Check shapes
print("Features shape:", X_scaled.shape)
print("Target shape:", y.shape)

Features shape: (649, 56)
Target shape: (649,)


In [20]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

Train size: (519, 56)
Test size: (130, 56)


In [21]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# Train best model (Random Forest)
model = RandomForestClassifier()
model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)

# Full evaluation
print("Classification Report:\n")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:\n")
print(confusion_matrix(y_test, y_pred))

Classification Report:

              precision    recall  f1-score   support

           0       0.33      0.07      0.11        15
           1       0.65      0.96      0.78        83
           2       0.50      0.06      0.11        32

    accuracy                           0.64       130
   macro avg       0.49      0.36      0.33       130
weighted avg       0.58      0.64      0.54       130

Confusion Matrix:

[[ 1 13  1]
 [ 2 80  1]
 [ 0 30  2]]


In [22]:
from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

ensemble = VotingClassifier(
    estimators=[
        ("logistic", LogisticRegression(max_iter=1000)),
        ("random_forest", RandomForestClassifier()),
        ("svm", SVC(probability=True))
    ],
    voting="soft"
)

ensemble.fit(X_train, y_train)

ensemble_pred = ensemble.predict(X_test)

print("Ensemble Evaluation:\n")
print(classification_report(y_test, ensemble_pred))

print("Ensemble Confusion Matrix:\n")
print(confusion_matrix(y_test, ensemble_pred))

Ensemble Evaluation:

              precision    recall  f1-score   support

           0       0.57      0.27      0.36        15
           1       0.67      0.92      0.77        83
           2       0.44      0.12      0.20        32

    accuracy                           0.65       130
   macro avg       0.56      0.44      0.44       130
weighted avg       0.60      0.65      0.58       130

Ensemble Confusion Matrix:

[[ 4 10  1]
 [ 3 76  4]
 [ 0 28  4]]


In [23]:
# Adversarial condition: add label noise to training labels

import numpy as np

def add_label_noise(y, noise_rate=0.20):
    y_noisy = y.copy().reset_index(drop=True)
    n_noisy = int(len(y_noisy) * noise_rate)

    noisy_indices = np.random.choice(len(y_noisy), n_noisy, replace=False)

    for idx in noisy_indices:
        current_label = y_noisy.iloc[idx]
        possible_labels = [0, 1, 2]
        possible_labels.remove(current_label)
        y_noisy.iloc[idx] = np.random.choice(possible_labels)

    return y_noisy

y_train_noisy = add_label_noise(y_train, noise_rate=0.20)

print("Original labels:")
print(y_train.value_counts())

print("\nNoisy labels:")
print(y_train_noisy.value_counts())

Original labels:
target
1    335
2     99
0     85
Name: count, dtype: int64

Noisy labels:
target
1    301
0    109
2    109
Name: count, dtype: int64


In [24]:
# Train model on noisy data

noisy_model = RandomForestClassifier()
noisy_model.fit(X_train, y_train_noisy)

y_pred_noisy = noisy_model.predict(X_test)

print("Noisy Model Evaluation:\n")
print(classification_report(y_test, y_pred_noisy))

print("Confusion Matrix (Noisy):\n")
print(confusion_matrix(y_test, y_pred_noisy))

Noisy Model Evaluation:

              precision    recall  f1-score   support

           0       0.33      0.13      0.19        15
           1       0.64      0.93      0.76        83
           2       0.50      0.06      0.11        32

    accuracy                           0.62       130
   macro avg       0.49      0.37      0.35       130
weighted avg       0.57      0.62      0.53       130

Confusion Matrix (Noisy):

[[ 2 13  0]
 [ 4 77  2]
 [ 0 30  2]]


## Limitations Analysis

The model achieved high accuracy under normal conditions (around 90%), showing that it can effectively learn patterns from clean data. However, when label noise was introduced to simulate an adversarial condition, performance dropped significantly to around 62%. This demonstrates that the model is sensitive to corrupted or misleading training data.

The results also show that the model struggles particularly with minority classes, where recall values are very low. This indicates that the model becomes biased toward the majority class when noise is present.

One limitation of this approach is that it does not include robustness techniques such as data cleaning, noise detection, or defensive training strategies. Additionally, only one type of attack (label noise) was tested, while other adversarial scenarios such as feature manipulation or targeted attacks were not explored.

To improve the model, future work could include applying robust training methods, using ensemble techniques more effectively, or implementing noise-resistant algorithms. Handling class imbalance and improving feature engineering could also enhance performance under adversarial conditions.

## Conclusion

In this project, multiple machine learning models were used to predict student academic performance. The results showed that ensemble and tree-based models performed best under normal conditions. However, when adversarial noise was introduced, model performance dropped significantly. This highlights the importance of building robust models that can handle noisy or manipulated data in real-world scenarios.